fig_zeno — Fig. `fig:zeno` (quantum Zeno crossover on the plaquette)
(a,b) loss coefficient kappa versus gamma/J from the mixed and Néel starts; (c) mixed: correlation-hole factor D/O,
(d) Néel: O = sum_r n_r,up n_r,dn, both as means over the kappa window (definitions in `postprocess.py`).
Reads `data/` (default) or a fresh `output/` (`USE_OUTPUT = True`) and writes `fig_zeno.pdf`.

In [1]:
USE_OUTPUT = False
OUTPUT_DIR = "output"

In [2]:
import sys
from pathlib import Path
import numpy as np
HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent))
from common import io as IO
from common.style import plt, W1, style, panel, tag, legend_handles, LABEL, save
if USE_OUTPUT:
    import postprocess
    D = postprocess.build(OUTPUT_DIR)
else:
    D = IO.read_data(HERE / "data")
K, WD = D["kappa.csv"], D["window.csv"]
MS = ["exact", "HF", "TPPX", "DSL"]
Z = {"exact": 2, "HF": 3, "TPPX": 3, "DSL": 4}
mk = lambda m: dict(marker="s" if m == "exact" else "o", ms=2.2)
def kap(ic, m):
    s = IO.select(K, ic=ic, method=m); o = np.argsort(s["gamma"]); return s["gamma"][o], s["kappa"][o]
def win(ic, m, q, gs):
    s = IO.select(WD, ic=ic, method=m); v = dict(zip(s["gamma"], s[q])); return [v[g] for g in gs]
G = sorted(IO.select(WD, ic="mixed", method="exact")["gamma"])

In [3]:
fig, axg = plt.subplots(2, 2, figsize=(W1, 4.2))
fig.subplots_adjust(left=0.175, right=0.985, top=0.88, bottom=0.10, hspace=0.45, wspace=0.55)
for j, (ic, lab) in enumerate((("mixed", "mixed"), ("neel", "Néel"))):
    a = axg[0, j]
    for m in MS:
        gs, v = kap(ic, m); a.loglog(gs, v, **mk(m), **style(m), zorder=Z[m])
    a.set_xlabel(r"$\gamma/J$"); a.set_ylabel(r"$\kappa/J$"); a.set_xlim(0.07, 50)
    tag(a, lab, "ul"); panel(a, "(a)" if j == 0 else "(b)")
axg[0, 0].set_ylim(2e-2, 20); axg[0, 1].set_ylim(2e-2, 8)
axg[0, 1].annotate("DSL diverges\nfor $\\gamma/J\\geq10$", xy=(7, 1.92), xytext=(0.97, 0.05), textcoords="axes fraction",
                   fontsize=7, ha="right", va="bottom", arrowprops=dict(arrowstyle="-", lw=0.5, color="#8a8984"))
a = axg[1, 0]
for m in MS:
    a.loglog(G, win("mixed", m, "D_over_O", G), **mk(m), **style(m), zorder=Z[m])
gg = np.logspace(0.9, np.log10(40), 10); a.loglog(gg, 12 / gg**2, color="#8a8984", lw=0.6, ls=(0, (1, 1.5)))
a.text(9, 12 / 9**2 * 0.45, r"$\propto(J/\gamma)^2$", fontsize=7, ha="right", va="top", color="#52514e")
a.set_xlabel(r"$\gamma/J$"); a.set_ylabel(r"$D/O$"); a.set_ylim(5e-4, 3); a.set_xlim(0.07, 50)
tag(a, "mixed", "ll"); panel(a, "(c)")
a = axg[1, 1]
for m in MS:   # DSL only where it does not diverge (gamma/J <= 7)
    gm = [g for g in G if m != "DSL" or g < 10]
    a.loglog(gm, win("neel", m, "O", gm), **mk(m), **style(m), zorder=Z[m])
a.loglog(gg, 8 / gg**2, color="#8a8984", lw=0.6, ls=(0, (1, 1.5)))
a.text(9, 8 / 9**2 * 0.45, r"$\propto(J/\gamma)^2$", fontsize=7, ha="right", va="top", color="#52514e")
a.set_xlabel(r"$\gamma/J$"); a.set_ylabel(r"$O$"); a.set_xlim(0.07, 50)
tag(a, "Néel", "ll"); panel(a, "(d)")
fig.legend(legend_handles(MS), [LABEL[m] for m in MS], loc="upper center", ncol=4, bbox_to_anchor=(0.55, 1.0),
           columnspacing=1.2)
fig.align_ylabels(axg[:, 0]); fig.align_ylabels(axg[:, 1])
save(fig, HERE / "fig_zeno")